# Módulo 01 · Aula 5: Chunking e indexação

**Duração:** 1h15

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- explicar por que documentos são divididos em trechos (*chunks*) antes de virar vetores;
- usar o `RecursiveCharacterTextSplitter` e o `MarkdownHeaderTextSplitter` (seções viram metadados);
- explicar o **cabeçalho de contexto** (título > seção > página) e medir o efeito dele na busca;
- escolher um `chunk_size` com um experimento: número de trechos, custo em tokens e acerto da busca;
- atualizar um índice sem recriá-lo, com **ids estáveis** (`add_chunks`, `remove_source`, `reindex_file`);
- usar as seções **2. Dividir** e **3. Indexar** de `src/rag/ingestion.py`, inclusive pela linha de comando.

## Pré-requisitos

- Aula 1.4: `Document`, metadados e `load_folder()`.
- Aulas 1.2 e 1.3: embeddings, similaridade de cosseno, `build_index`, `save_index` e `load_index`.

## O que vamos construir

O resto do caminho entre os documentos carregados na aula passada e o índice:

```mermaid
flowchart LR
    D[Document] --> T{Markdown?}
    T -->|sim| H[MarkdownHeaderTextSplitter<br/>seções viram metadados]
    T -->|não| R
    H --> R[RecursiveCharacterTextSplitter<br/>chunk_size + chunk_overlap]
    R --> C[cabeçalho de contexto] --> ID[id estável]
    ID --> E[embeddings] --> I[(FAISS)]
    ID -.já existe no índice?.-> I
```

> 💰 **Custo desta aula:** cerca de 1.300 trechos e 30 perguntas passam pelo modelo de embeddings (índices
> pequenos, em memória ou em uma pasta temporária). No preço atual do `text-embedding-3-large` isso fica em
> poucos centavos de dólar. O índice principal em `data/faiss_index` é só lido, nunca alterado.

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. Por que dividir?

Na aula passada vimos que uma FAQ tem cerca de 230 caracteres e uma política, quase 5.000. Um embedding é **um
vetor por texto**, não importa o tamanho. Se um manual inteiro vira um vetor, esse vetor é uma espécie de "média"
de todos os assuntos do manual (pareamento, bateria, limpeza, garantia) e fica distante de qualquer pergunta
específica. Vamos medir com uma pergunta real, comparando o manual inteiro, uma página e só o parágrafo que
responde (4 embeddings):

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.models import get_embeddings
from src.rag.ingestion import load_file, load_folder

DOCS = Path(settings.documents_dir)
embeddings = get_embeddings()

buds = load_file(DOCS / "manuals/fn-001_nova_buds.pdf", base=DOCS)
whole_manual = "\n".join(d.page_content for d in buds)
page_3 = buds[2].page_content
battery = page_3[page_3.index("5. Bateria") : page_3.index("Indicações do LED")]

question = "Quanto dura a bateria do Fone Nova Buds com o cancelamento de ruído ligado?"
texts = {"manual inteiro": whole_manual, "página 3": page_3, "parágrafo da bateria": battery}

q = np.array(embeddings.embed_query(question))
vectors = np.array(embeddings.embed_documents(list(texts.values())))
scores = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))

for (name, text), score in zip(texts.items(), scores):
    print(f"{name:<22} {len(text):>6} caracteres   similaridade {score:.3f}")

2026-09-27 22:58:55 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


manual inteiro          10393 caracteres   similaridade 0.596
página 3                 2127 caracteres   similaridade 0.633
parágrafo da bateria      462 caracteres   similaridade 0.699


**Observe:** quanto mais focado o texto, mais parecido ele fica com a pergunta. Três motivos para dividir:

1. **Busca mais precisa:** trechos com um assunto só geram vetores que "apontam" para esse assunto.
2. **Contexto menor e mais barato:** o modelo recebe os 4 trechos mais relevantes, não 4 documentos inteiros.
3. **Citação precisa:** "Manual do Nova Buds, página 3" ajuda mais que "Manual do Nova Buds".

O risco oposto também existe: um trecho pequeno demais perde o contexto ("Cada fone oferece até 6 horas"...
qual fone?). Dividir bem é equilibrar esses dois lados.

## 2. `RecursiveCharacterTextSplitter`: separadores, tamanho e sobreposição

O splitter padrão do LangChain (pacote `langchain-text-splitters`) tem três parâmetros principais:

| Parâmetro | Valor no projeto | Significado |
|---|---|---|
| `chunk_size` | 1000 | tamanho máximo do trecho, em **caracteres** (`len`) |
| `chunk_overlap` | 150 | quanto do fim de um trecho se repete no começo do próximo |
| `separators` | `["\n\n", "\n", ". ", " ", ""]` | onde cortar, do melhor para o pior |

O algoritmo é **recursivo**: corta pelo primeiro separador (parágrafos); o pedaço que ainda passar do tamanho é
cortado pelo próximo (linhas), depois frases, palavras e caracteres. Depois junta vizinhos até encher o `chunk_size`.

Visualizamos os cortes na página 3 do manual do Nova Buds, com trechos de 500 caracteres e a sobreposição de 150 do
projeto (`add_start_index=True` grava onde cada trecho começa; o que se repete do anterior aparece entre `«»`). No fim,
comparamos onde os trechos terminam com os separadores do projeto e só com `""` (corte por caractere):

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def show_chunks(pieces) -> None:
    """Mostra cada trecho com a sua posição e marca entre «» o que se repete do trecho anterior."""
    previous_end = 0
    for number, piece in enumerate(pieces, start=1):
        start = piece.metadata["start_index"]
        end = start + len(piece.page_content)
        overlap = max(0, previous_end - start)
        text = piece.page_content
        print(f"--- trecho {number}: caracteres {start} a {end} ({len(text)}), sobreposição {overlap}")
        print(f"«{text[:overlap]}»{text[overlap:]}" if overlap else text)
        previous_end = end


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=150, separators=SEPARATORS, add_start_index=True
)
show_chunks(splitter.create_documents([page_3]))

print("\n" + "=" * 100)
for name, separators in [("separadores do projeto", SEPARATORS), ('só "" (caracteres)', [""])]:
    cutter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, separators=separators)
    print(f"Fim de cada trecho, {name}:", [chunk[-18:] for chunk in cutter.split_text(page_3)])

--- trecho 1: caracteres 0 a 475 (475), sobreposição 0
TechNova | Fone Nova Buds (FN-001) | Manual do usuário
4. Primeiros passos
4.1 Primeira carga
Retire as películas de proteção dos contatos, coloque os fones no estojo e carregue até o LED ficar verde fixo
antes do primeiro uso.
4.2 Pareamento
- Abra o estojo com os fones dentro.
- Segure o botão do estojo por 3 segundos até o LED piscar branco. O modo de pareamento fica ativo por
3 minutos.
- No celular, ative o Bluetooth e selecione "Nova Buds" na lista de dispositivos.
--- trecho 2: caracteres 383 a 821 (438), sobreposição 92
«3 minutos.
- No celular, ative o Bluetooth e selecione "Nova Buds" na lista de dispositivos.»
- Quando a conexão for concluída, o LED fica branco fixo por 5 segundos e você ouve um sinal sonoro nos
fones.
4.3 Controles por toque
Gesto Função (padrão de fábrica)
1 toque (qualquer fone) Reproduzir ou pausar; atender chamada
2 toques no fone direito Próxima faixa; encerrar chamada
2 toques no fone esquerdo Fai

**Observe:**

- Os cortes caem em **fim de linha**: o texto do PDF não tem parágrafos (`\n\n`), então o segundo separador
  (`\n`) é o que funciona. Nenhuma palavra foi cortada ao meio. Com `""`, os trechos terminam no meio de
  palavras, o que gera vetores piores e citações estranhas.
- A sobreposição fica **abaixo de 150**: ela é feita com pedaços inteiros (linhas), repetindo as últimas linhas
  que cabem em 150 caracteres. `chunk_overlap` é um máximo, não um valor exato.
- Graças à sobreposição, uma informação perto de um corte aparece inteira em pelo menos um trecho. Sem ela, a
  frase da fronteira pode ficar dividida entre dois trechos e não ser encontrada por nenhum deles.

> ⚠️ **Detalhe do separador `". "`:** por padrão (`keep_separator=True`) o separador fica no **começo** do
> trecho seguinte. Quando o corte acontece entre frases, o próximo trecho começa com `". "`. É só cosmético;
> `keep_separator="end"` deixaria o ponto no fim do trecho anterior.

## 3. `MarkdownHeaderTextSplitter`: seções viram metadados

Documentos com estrutura merecem ser divididos **pela estrutura**. As políticas têm títulos `##` (seção) e `###`
(subseção). O `MarkdownHeaderTextSplitter` corta nesses títulos e grava o título de cada parte nos metadados.
Depois, as seções grandes passam pelo splitter recursivo, e os metadados de cada seção seguem para os trechos
(aqui com trechos de 400 caracteres, para ver o efeito):

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter

exchange = load_file(DOCS / "policies/exchange.md", base=DOCS)[0]

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("##", "section"), ("###", "subsection")],
    strip_headers=False,  # mantém a linha "## Título" no texto
)
sections = header_splitter.split_text(exchange.page_content)
for part in sections:
    print(f"{len(part.page_content):>4} caracteres  {part.metadata}")

# Segunda etapa: o splitter recursivo nas seções, levando os metadados junto.
section_docs = [
    Document(page_content=part.page_content, metadata={**exchange.metadata, **part.metadata})
    for part in sections
]
small_chunks = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=60, separators=SEPARATORS
).split_documents(section_docs)
print(f"\n{len(sections)} seções -> {len(small_chunks)} trechos de até 400 caracteres")

continuation = next(c for c in small_chunks if not c.page_content.startswith("#"))
print("\nUm trecho que continua uma seção (sem o título no texto):")
print({k: continuation.metadata[k] for k in ("title", "section")})
print(continuation.page_content[:160], "...")

 268 caracteres  {}
 456 caracteres  {'section': 'Resumo da regra'}
 477 caracteres  {'section': 'Direito de arrependimento e prazo ampliado'}
 605 caracteres  {'section': 'Condições para a troca'}
 333 caracteres  {'section': 'Condições para a troca', 'subsection': 'Produtos que não podem ser trocados por arrependimento'}
 712 caracteres  {'section': 'Como solicitar uma troca'}
 265 caracteres  {'section': 'Como solicitar uma troca', 'subsection': 'Coleta em domicílio'}
 694 caracteres  {'section': 'Troca por outro modelo ou outra cor'}
 564 caracteres  {'section': 'Troca de produto com defeito'}
 715 caracteres  {'section': 'Exemplos práticos'}
 163 caracteres  {'section': 'Dúvidas'}

11 seções -> 20 trechos de até 400 caracteres

Um trecho que continua uma seção (sem o título no texto):
{'title': 'Política de Troca', 'section': 'Resumo da regra'}
- Produtos com **defeito** podem ser trocados durante **toda a garantia** (12 meses a partir da entrega), após avaliação da assistência té

**Observe:**

- A política virou 11 partes, uma por seção ou subseção. A introdução, antes do primeiro `##`, fica sem
  `section`, e a subseção carrega também a seção mãe.
- Nenhuma parte mistura duas seções: um trecho sobre "Coleta em domicílio" não termina com o começo de "Troca por
  outro modelo".
- O trecho de continuação não tem o título `## ...` no texto, mas sabe a que seção pertence pelos metadados. Isso
  é a base da próxima ideia.

## 4. O cabeçalho de contexto

Um trecho do meio de um documento raramente diz de onde veio. A função `split_documents` do projeto (que junta as
duas etapas acima; detalhes na seção 9) tem a opção `add_header`. Veja o mesmo trecho do manual do Nova Buds
sem e com o **cabeçalho de contexto**, montado com os metadados por `context_header`:

In [5]:
from src.rag.ingestion import context_header, split_documents

documents = load_folder()
manuals = [d for d in documents if d.metadata["type"] == "pdf"]
manual_chunks_plain = split_documents(manuals, add_header=False)
manual_chunks_header = split_documents(manuals)  # add_header=True é o padrão


def buds_battery(chunks: list[Document]) -> Document:
    return next(
        c for c in chunks
        if c.metadata["source"].endswith("fn-001_nova_buds.pdf") and "Cada fone oferece" in c.page_content
    )


plain = buds_battery(manual_chunks_plain)
print("SEM cabeçalho:\n" + plain.page_content[:300], "...")
print("\nO texto menciona 'Nova Buds'?", "Nova Buds" in plain.page_content)
print("\nCOM cabeçalho:\n" + buds_battery(manual_chunks_header).page_content[:160], "...")

print("\nOutros exemplos de context_header:")
print(" ", context_header({"title": "Política de Troca", "section": "Como solicitar uma troca",
                           "subsection": "Coleta em domicílio"}))
print(" ", context_header({"title": "Como rastrear seu pedido | Central de Ajuda TechNova"}))

2026-09-27 22:58:56 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=34 | documentos=105


2026-09-27 22:58:56 | INFO     | src.rag.ingestion | Documentos divididos | documentos=40 | trechos=106


2026-09-27 22:58:56 | INFO     | src.rag.ingestion | Documentos divididos | documentos=40 | trechos=106


SEM cabeçalho:
Tocar e segurar 2 s (qualquer fone) Alternar ANC, modo ambiente e desligado; durante uma chamada recebida,
recusar
Os fones pausam a música automaticamente quando você retira um deles do ouvido e retomam ao
recolocá-lo (detecção de uso, que pode ser desligada no app). Fora do estojo e sem conexão, o ...

O texto menciona 'Nova Buds'? False

COM cabeçalho:
[Manual do usuário: Fone Nova Buds > página 3]
Tocar e segurar 2 s (qualquer fone) Alternar ANC, modo ambiente e desligado; durante uma chamada recebida,
recusa ...

Outros exemplos de context_header:
  Política de Troca > Como solicitar uma troca > Coleta em domicílio
  Como rastrear seu pedido | Central de Ajuda TechNova


Sem cabeçalho, o trecho fala de "cada fone" e de "ANC", mas não diz **qual** fone; o Headset Nova Studio também
tem bateria e ANC. Como o cabeçalho entra no `page_content`, ele entra também no vetor.

Isso melhora a busca? Montamos dois índices pequenos **só com os manuais** (cerca de 100 trechos cada, em
memória), um sem e outro com cabeçalho, e buscamos as 6 perguntas de `data/eval/questions.json` sobre manuais.
Contamos quantos dos 4 trechos devolvidos vêm do manual certo.

> 💡 Como vimos na aula 1.2, `embed_documents` manda vários textos em uma única requisição. Calculamos os
> vetores das perguntas uma vez e reaproveitamos nos dois índices.

In [6]:
from src.rag.ingestion import estimate_tokens
from src.rag.vectorstore import build_index

questions = json.loads(Path(settings.eval_questions_path).read_text(encoding="utf-8"))
manual_questions = [q for q in questions if q["id"] in {"q04", "q05", "q06", "q07", "q12", "q18"}]
manual_vectors = embeddings.embed_documents([q["question"] for q in manual_questions])

index_plain = build_index(manual_chunks_plain)
index_header = build_index(manual_chunks_header)


def correct_in_top_k(index, vector, expected: list[str], k: int = 4) -> tuple[int, list[str]]:
    """Quantos dos k trechos vêm de uma fonte esperada, e quais fontes erradas apareceram."""
    sources = [d.metadata["source"] for d in index.similarity_search_by_vector(vector, k=k)]
    wrong = [s.split("/")[-1] for s in sources if s not in expected]
    return len(sources) - len(wrong), wrong


rows = []
for q, vector in zip(manual_questions, manual_vectors):
    plain_ok, plain_wrong = correct_in_top_k(index_plain, vector, q["expected_sources"])
    header_ok, _ = correct_in_top_k(index_header, vector, q["expected_sources"])
    rows.append({"id": q["id"], "question": q["question"][:50], "no_header": f"{plain_ok}/4",
                 "with_header": f"{header_ok}/4", "intruders_no_header": ", ".join(plain_wrong)})

extra = estimate_tokens(manual_chunks_header) / estimate_tokens(manual_chunks_plain) - 1
print(f"Custo do cabeçalho: +{extra:.0%} de tokens para indexar os manuais")
pd.DataFrame(rows)

2026-09-27 22:58:56 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=106 | embeddings=azure · text-embedding-3-large


2026-09-27 22:58:58 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=106 | embeddings=azure · text-embedding-3-large


Custo do cabeçalho: +6% de tokens para indexar os manuais


,id,question,no_header,with_header,intruders_no_header
0,q04,Liguei o Monitor Nova View 27 pelo HDMI e não ...,4/4,4/4,
1,q05,Quanto dura a bateria do Fone Nova Buds com o ...,3/4,4/4,fn-002_nova_studio.pdf
2,q06,Posso mergulhar a caixa de som Nova Boom? Até ...,4/4,4/4,
3,q07,De quantos watts é o carregamento rápido do No...,3/4,4/4,fn-002_nova_studio.pdf
4,q12,O Smartphone Nova X é à prova d'água?,3/4,4/4,sm-004_nova_mini.pdf
5,q18,Como conecto o Nova Buds no meu celular pela p...,4/4,4/4,


**Observe:** sem cabeçalho, trechos de **outros** manuais entram no top 4 (fones e smartphones têm textos
parecidos sobre bateria, carregamento e água). Com o cabeçalho, o nome do produto está em todo trecho e os
intrusos saem, por alguns tokens a mais. Na hora de gerar a resposta, um trecho do manual errado é pior que
nenhum: o modelo pode misturar a bateria de um produto com a de outro.

## 5. Ids estáveis

Cada trecho recebe um id em `metadata["id"]`, calculado por `chunk_id`: um hash SHA-1 da fonte, da página (ou
linha do CSV) e do texto do trecho. É o que vai permitir **atualizar o índice sem recriá-lo**, porque:

- o mesmo trecho gera **sempre** o mesmo id, em qualquer execução e em qualquer máquina;
- **qualquer** mudança no texto (ou na fonte) gera outro id.

In [7]:
from src.rag.ingestion import chunk_id

chunk = manual_chunks_header[10]
print("id guardado:           ", chunk.metadata["id"])
print("recalculado:           ", chunk_id(chunk))

edited = Document(page_content=chunk.page_content.replace("a", "á", 1), metadata=chunk.metadata)
print("com 1 letra trocada:   ", chunk_id(edited))

moved = Document(page_content=chunk.page_content, metadata={**chunk.metadata, "source": "manuals/outro.pdf"})
print("mesmo texto, outra fonte:", chunk_id(moved))

again = split_documents(manuals)
same = [a.metadata["id"] for a in again] == [c.metadata["id"] for c in manual_chunks_header]
print("\nDividir de novo gera exatamente os mesmos ids?", same)

id guardado:            ada6fe95de49e8cc
recalculado:            ada6fe95de49e8cc
com 1 letra trocada:    d1f28328fcbefa6c
mesmo texto, outra fonte: 671a35e4df8a10a6
2026-09-27 22:58:59 | INFO     | src.rag.ingestion | Documentos divididos | documentos=40 | trechos=106



Dividir de novo gera exatamente os mesmos ids? True


> 💡 Por que não um `uuid4()`? Ele muda a cada execução: rodar a ingestão duas vezes duplicaria tudo, e não
> haveria como saber quais trechos o índice já tem. Com o hash do conteúdo, "já está no índice?" vira uma
> consulta de id.

## 6. Experimento: qual `chunk_size`?

Não existe tamanho certo universal: depende dos documentos e das perguntas. O jeito honesto de escolher é
**medir**. Comparamos 400, 1000 e 2000 caracteres em três dimensões:

1. **Tamanho do índice:** número de trechos e tokens enviados ao modelo de embeddings (custo de indexar).
2. **Qualidade da busca:** nas 26 perguntas de `questions.json` que têm resposta na base, a fonte certa aparece
   no top 4? Em primeiro lugar? Nas 6 perguntas `multi_document`, **todas** as fontes esperadas aparecem?
3. **Custo por pergunta:** quantos tokens somam os 4 trechos recuperados (é o que vai no prompt, a cada pergunta).

O índice de 1000 **já existe**: é o índice principal em `data/faiss_index`, criado com a configuração padrão.
Graças aos ids estáveis, conferimos que ele tem exatamente os mesmos trechos e o reaproveitamos. Os de 400 e 2000
são criados em memória:

In [8]:
from src.rag.ingestion import index_folder
from src.rag.vectorstore import index_exists, load_index, stored_chunks

SIZES = [400, 1000, 2000]
chunks_by_size = {size: split_documents(documents, chunk_size=size) for size in SIZES}

if not index_exists():
    index_folder()  # cria data/faiss_index na primeira vez

main_index = load_index()
same_chunks = set(stored_chunks(main_index)) == {c.metadata["id"] for c in chunks_by_size[1000]}
print("O índice principal tem exatamente os trechos de chunk_size=1000?", same_chunks)

indexes = {1000: main_index if same_chunks else build_index(chunks_by_size[1000])}
for size in (400, 2000):
    indexes[size] = build_index(chunks_by_size[size])

2026-09-27 22:58:59 | INFO     | src.rag.ingestion | Documentos divididos | documentos=105 | trechos=750


2026-09-27 22:58:59 | INFO     | src.rag.ingestion | Documentos divididos | documentos=105 | trechos=321


2026-09-27 22:58:59 | INFO     | src.rag.ingestion | Documentos divididos | documentos=105 | trechos=237


O índice principal tem exatamente os trechos de chunk_size=1000? True
2026-09-27 22:58:59 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=750 | embeddings=azure · text-embedding-3-large


2026-09-27 22:59:01 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=237 | embeddings=azure · text-embedding-3-large


In [9]:
answerable = [q for q in questions if q["type"] != "no_answer"]
question_vectors = embeddings.embed_documents([q["question"] for q in answerable])


def evaluate(index, k: int = 4) -> pd.DataFrame:
    """Uma linha por pergunta: a fonte certa apareceu? em 1º? todas as fontes? quantos tokens de contexto?"""
    rows = []
    for q, vector in zip(answerable, question_vectors):
        found = index.similarity_search_by_vector(vector, k=k)
        sources = [d.metadata["source"] for d in found]
        expected = q["expected_sources"]
        rows.append({
            "type": q["type"], "hit": any(s in expected for s in sources), "top1": sources[0] in expected,
            "all_sources": all(e in sources for e in expected), "context_tokens": estimate_tokens(found),
        })
    return pd.DataFrame(rows)


summary = {}
for size in SIZES:
    r = evaluate(indexes[size])
    multi = r[r["type"] == "multi_document"]
    summary[size] = {
        "trechos no índice": len(chunks_by_size[size]),
        "tokens para indexar": estimate_tokens(chunks_by_size[size]),
        "fonte certa no top 4": f"{r['hit'].sum()}/{len(r)}",
        "fonte certa em 1º": f"{r['top1'].sum()}/{len(r)}",
        "multi_document completas": f"{multi['all_sources'].sum()}/{len(multi)}",
        "tokens de contexto por pergunta": round(r["context_tokens"].mean()),
    }
summary = pd.DataFrame(summary)
summary.columns.name = "chunk_size"
summary

chunk_size,400,1000,2000
trechos no índice,750,321,237
tokens para indexar,74073,56705,52876
fonte certa no top 4,23/26,25/26,25/26
fonte certa em 1º,13/26,15/26,12/26
multi_document completas,3/6,2/6,4/6
tokens de contexto por pergunta,396,783,1179


**Como ler o resultado:**

- **400:** o índice fica maior e mais caro de montar (a sobreposição e o cabeçalho se repetem em cada trecho), e
  trechos curtos perdem contexto: é o tamanho que mais erra a fonte. Em compensação, manda menos tokens ao modelo.
- **2000:** índice menor e mais barato; trechos grandes às vezes "pegam" as duas fontes de uma pergunta que
  combina documentos. Mas cada pergunta leva cerca de 50% mais tokens de contexto que com 1000, e trechos longos
  misturam assuntos, o que aumenta o ruído entregue ao modelo.
- **1000** fica no meio: acerta a fonte no top 4 tanto quanto 2000, é o que mais acerta em primeiro lugar e
  manda cerca de um terço a menos de contexto que 2000 (em troca, é o que menos completa as `multi_document`, 2 de 6). É o padrão do projeto (`CHUNK_SIZE=1000` e
  `CHUNK_OVERLAP=150` em `src/config.py`).

> ⚠️ São 26 perguntas: uma diferença de uma ou duas é sinal, não prova. No módulo 03 medimos a recuperação com
> métricas completas. O importante é o método: **mudar um parâmetro por vez, medir e só então decidir**. E mudar
> o `chunk_size` exige recriar o índice.

## 7. O pipeline completo: `index_folder()`

`index_folder(folder, target)` faz tudo de uma vez: `load_folder` -> `split_documents` -> `build_index` ->
`save_index`. Antes de gastar, estime o custo com `estimate_tokens`, que conta os tokens sem chamar a API.

A base inteira já está em `data/faiss_index`, e esse índice não deve ser sobrescrito por experimentos. Para praticar,
rodamos o pipeline em uma **cópia das políticas**, em uma pasta temporária, mantendo a subpasta `policies/` para que o
`source` dos trechos seja o mesmo da base real (`policies/exchange.md`):

In [10]:
import shutil
import tempfile

all_chunks = chunks_by_size[1000]
all_tokens = estimate_tokens(all_chunks)
price_per_million = 0.13  # US$ por 1 milhão de tokens do text-embedding-3-large (confira a tabela atual)
cost = all_tokens / 1e6 * price_per_million
print(f"Base inteira: {len(all_chunks)} trechos, ~{all_tokens:_} tokens".replace("_", "."),
      f"-> ~US$ {cost:.4f} para indexar".replace(".", ","))

WORK_DIR = Path(tempfile.mkdtemp(prefix="technova_01_05_"))
docs_copy = WORK_DIR / "docs"
index_dir = WORK_DIR / "index"
shutil.copytree(DOCS / "policies", docs_copy / "policies")

report = index_folder(docs_copy, target=index_dir)
print("\n", report)

print("\nArquivos do índice:")
for path in sorted(index_dir.iterdir()):
    print(f"  {path.name:<16} {path.stat().st_size:>9,} bytes".replace(",", "."))
print("\nindex_info.json:", json.loads((index_dir / "index_info.json").read_text(encoding="utf-8")))

Base inteira: 321 trechos, ~56.705 tokens -> ~US$ 0,0074 para indexar
2026-09-27 22:59:02 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=8 | documentos=8


2026-09-27 22:59:02 | INFO     | src.rag.ingestion | Documentos divididos | documentos=8 | trechos=88


2026-09-27 22:59:02 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=88 | embeddings=azure · text-embedding-3-large


2026-09-27 22:59:03 | INFO     | src.rag.vectorstore | Índice salvo | pasta=/tmp/technova_01_05_ygyhpngh/index | trechos=88



 {'files': 8, 'documents': 8, 'chunks': 88, 'estimated_tokens': 10794}

Arquivos do índice:
  index.faiss      1.081.389 bytes
  index.pkl           59.161 bytes
  index_info.json        130 bytes

index_info.json: {'embeddings': 'azure · text-embedding-3-large', 'chunks': 88, 'dimensions': 3072, 'updated_at': '2026-09-27T22:59:03'}


O `index_info.json` registra o modelo de embeddings e a dimensão dos vetores. Como vimos na aula 1.2, **um índice
só funciona com o modelo que o criou**: `load_index` compara o registro com `describe_embeddings()` e avisa quando
eles não batem. Trocar de modelo, de dimensão ou de `chunk_size` exige rodar `index_folder` de novo.

## 8. Atualização incremental

Documentos mudam: uma política é revisada, um manual novo chega, um artigo é retirado. Recriar o índice a cada
mudança custa tempo e dinheiro. Com os ids estáveis, o projeto oferece três operações:

- `add_chunks(index, chunks)`: adiciona só os trechos cujo id ainda **não** está no índice;
- `remove_source(index, source)`: remove todos os trechos de uma fonte;
- `reindex_file(index, path, base)`: remove a versão antiga de um arquivo e adiciona a nova.

### 8.1 `add_chunks` e `remove_source`

In [11]:
from src.rag.ingestion import add_chunks, reindex_file, remove_source
from src.rag.vectorstore import save_index

index = load_index(index_dir)
policy_chunks = split_documents(load_folder(docs_copy))
print("Trechos no índice:", index.index.ntotal)

print("\nadd_chunks com os MESMOS trechos -> novos:", add_chunks(index, policy_chunks))

print("\nremove_source(shipping_2024) -> removidos:", remove_source(index, "policies/shipping_2024.md"))
print("Trechos no índice:", index.index.ntotal)

print("\nadd_chunks de novo -> novos:", add_chunks(index, policy_chunks))
print("Trechos no índice:", index.index.ntotal)

2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=8 | documentos=8


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos divididos | documentos=8 | trechos=88


Trechos no índice: 88
2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Trechos adicionados | novos=0 | ignorados=88



add_chunks com os MESMOS trechos -> novos: 0
2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Trechos removidos | fonte=policies/shipping_2024.md | removidos=13



remove_source(shipping_2024) -> removidos: 13
Trechos no índice: 75


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Trechos adicionados | novos=13 | ignorados=75



add_chunks de novo -> novos: 13
Trechos no índice: 88


Rodar a ingestão duas vezes não duplica nada, e só os 13 trechos que faltavam passaram pelo modelo de
embeddings.

### 8.2 `reindex_file`: uma política foi editada

Simulamos uma revisão da política de troca: copiamos o arquivo para outra pasta temporária e mudamos o prazo de
30 para 45 dias (os arquivos de `data/documents` nunca são alterados). Antes de reindexar, comparamos os ids da
versão antiga e da nova:

In [12]:
edited_dir = WORK_DIR / "edited"
(edited_dir / "policies").mkdir(parents=True)
original_text = (docs_copy / "policies/exchange.md").read_text(encoding="utf-8")
edited_text = original_text.replace("até 30 dias corridos após a entrega", "até 45 dias corridos após a entrega", 1)
(edited_dir / "policies/exchange.md").write_text(edited_text, encoding="utf-8")

old_ids = {c.metadata["id"] for c in split_documents(load_file(docs_copy / "policies/exchange.md", docs_copy))}
new_ids = {c.metadata["id"] for c in split_documents(load_file(edited_dir / "policies/exchange.md", edited_dir))}
print(f"Trechos: {len(old_ids)} | iguais nas duas versões: {len(old_ids & new_ids)} | "
      f"só na antiga: {len(old_ids - new_ids)} | só na nova: {len(new_ids - old_ids)}")

print("\n", reindex_file(index, edited_dir / "policies/exchange.md", base=edited_dir))

print("\nBusca: 'Em quantos dias posso trocar um produto?'")
for d in index.similarity_search("Em quantos dias posso trocar um produto?", k=2):
    rule = next((line for line in d.page_content.splitlines() if "dias" in line), "")
    print(f"  [{d.metadata['source']} > {d.metadata.get('section')}] {rule[:90]}")

2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos divididos | documentos=1 | trechos=11


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos divididos | documentos=1 | trechos=11


Trechos: 11 | iguais nas duas versões: 10 | só na antiga: 1 | só na nova: 1
2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos divididos | documentos=1 | trechos=11


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Trechos removidos | fonte=policies/exchange.md | removidos=11


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Trechos adicionados | novos=11 | ignorados=0



 {'source': 'policies/exchange.md', 'removed': 11, 'added': 11}

Busca: 'Em quantos dias posso trocar um produto?'


  [policies/exchange.md > Resumo da regra] - Trocas podem ser solicitadas em **até 45 dias corridos após a entrega** do pedido.
  [policies/exchange.md > Direito de arrependimento e prazo ampliado] O Código de Defesa do Consumidor garante 7 dias, a partir da entrega, para desistir de com


**Observe:** mudar uma frase alterou o id de **um** trecho só; os outros continuam iguais. O `reindex_file` é
simples de propósito: remove todos os trechos da fonte e adiciona todos de novo (11 embeddings). Uma versão mais
econômica calcularia só os trechos cujo id mudou. Para uma política de 11 trechos a diferença é irrelevante; para
um manual de 300 páginas, não.

### 8.3 Cuidado: adicionar não é sincronizar

O projeto também tem `index_incremental(folder, target)`, que carrega o índice salvo, adiciona os trechos novos
de uma pasta e salva. Ele é feito para **documentos novos**. Veja o que acontece se usarmos para "atualizar" a
pasta de políticas depois da edição:

In [13]:
from src.rag.ingestion import index_incremental

save_index(index, index_dir)
print("\n", index_incremental(docs_copy, target=index_dir))

index = load_index(index_dir)
print("\nBusca: 'Em quantos dias posso trocar um produto?'")
for d in index.similarity_search("Em quantos dias posso trocar um produto?", k=2):
    rule = next((line for line in d.page_content.splitlines() if "dias" in line), "")
    print(f"  [{d.metadata['source']} > {d.metadata.get('section')}] {rule[:90]}")

2026-09-27 22:59:03 | INFO     | src.rag.vectorstore | Índice salvo | pasta=/tmp/technova_01_05_ygyhpngh/index | trechos=88


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=8 | documentos=8


2026-09-27 22:59:03 | INFO     | src.rag.ingestion | Documentos divididos | documentos=8 | trechos=88


2026-09-27 22:59:04 | INFO     | src.rag.ingestion | Trechos adicionados | novos=1 | ignorados=87


2026-09-27 22:59:04 | INFO     | src.rag.vectorstore | Índice salvo | pasta=/tmp/technova_01_05_ygyhpngh/index | trechos=89



 {'chunks_read': 88, 'added': 1, 'total_in_index': 89}

Busca: 'Em quantos dias posso trocar um produto?'


  [policies/exchange.md > Resumo da regra] - Trocas podem ser solicitadas em **até 30 dias corridos após a entrega** do pedido.
  [policies/exchange.md > Resumo da regra] - Trocas podem ser solicitadas em **até 45 dias corridos após a entrega** do pedido.


Agora o índice tem as **duas versões** da regra, 30 e 45 dias, e o assistente pode citar qualquer uma. O
`index_incremental` só adiciona: ele nunca remove trechos que deixaram de existir. A regra de uso:

| Situação | Operação |
|---|---|
| documentos novos | `index_incremental(folder)` ou `add_chunks` |
| um arquivo foi editado | `reindex_file(index, path)` |
| um arquivo foi retirado | `remove_source(index, source)` |
| mudou o modelo de embeddings, o `chunk_size` ou muita coisa ao mesmo tempo | `index_folder()` (recria tudo) |

Corrigimos com `reindex_file` usando a versão original:

In [14]:
print(reindex_file(index, docs_copy / "policies/exchange.md", base=docs_copy))
texts_in_index = [c.page_content for c in stored_chunks(index).values()]
print("Trechos com '45 dias corridos':", sum("45 dias corridos" in t for t in texts_in_index))
print("Trechos no índice:", index.index.ntotal)

2026-09-27 22:59:04 | INFO     | src.rag.ingestion | Documentos divididos | documentos=1 | trechos=11


2026-09-27 22:59:04 | INFO     | src.rag.ingestion | Trechos removidos | fonte=policies/exchange.md | removidos=12


2026-09-27 22:59:04 | INFO     | src.rag.ingestion | Trechos adicionados | novos=11 | ignorados=0


{'source': 'policies/exchange.md', 'removed': 12, 'added': 11}
Trechos com '45 dias corridos': 0
Trechos no índice: 88


## 9. Do notebook para o projeto: [src/rag/ingestion.py](../src/rag/ingestion.py)

Usamos as seções **2. Dividir** e **3. Indexar** do arquivo ao longo da aula. Resumindo:

| Nome | O que faz |
|---|---|
| `MARKDOWN_HEADERS` | `[("##", "section"), ("###", "subsection")]`: os títulos que viram metadados |
| `make_splitter(chunk_size, chunk_overlap)` | o `RecursiveCharacterTextSplitter` com os separadores do projeto |
| `context_header(metadata)` | título > seção > subseção > página |
| `chunk_id(chunk)` | id estável: hash de fonte, página ou linha e texto |
| `split_documents(documents, chunk_size, chunk_overlap, add_header)` | título -> recursivo -> cabeçalho -> id |
| `estimate_tokens(chunks)` | tokens aproximados, para estimar o custo antes de indexar |
| `index_folder(folder, target)` | recria o índice do zero e salva |
| `add_chunks`, `remove_source`, `reindex_file` | atualização por id e por fonte |
| `index_incremental(folder, target)` | adiciona os documentos de uma pasta ao índice salvo |
| `main()` | a linha de comando `python -m src.rag.ingestion` |

Decisões de design:

- **Só Markdown passa pela etapa de títulos.** PDF já vem dividido por página e HTML tem poucos títulos. A FAQ
  nunca é dividida na prática: cada linha tem menos de 400 caracteres.
- **O cabeçalho vai no `page_content`**, para entrar no vetor, e **o id é calculado depois dele**: se o título
  de um documento mudar, os ids mudam também.
- `chunk_size` e `chunk_overlap` vêm de `settings` (`CHUNK_SIZE` e `CHUNK_OVERLAP` no `.env`), com a opção de
  sobrescrever por parâmetro, como no experimento.
- `add_chunks` descobre o que já existe com `stored_chunks`, de `src/rag/vectorstore.py`: o acesso ao armazenamento
  interno do FAISS fica isolado no mesmo arquivo que isola o `langchain-community`.

O mesmo arquivo roda como programa. É assim que o índice principal foi criado e é assim que, no projeto final,
documentos novos entram no índice:

```bash
python -m src.rag.ingestion                                # recria data/faiss_index a partir de data/documents
python -m src.rag.ingestion data/documents --incremental   # depois de mover os arquivos novos para a subpasta certa
```

A célula abaixo mostra a base inteira com a configuração padrão e só pede a ajuda da linha de comando, para não mexer
no índice principal:

In [15]:
import subprocess
import sys

print(f"chunk_size={settings.chunk_size}, chunk_overlap={settings.chunk_overlap}")
print(f"{len(documents)} documentos -> {len(all_chunks)} trechos")
print(pd.Series([c.metadata["type"] for c in all_chunks]).value_counts().to_string())

example = next(c for c in all_chunks if c.metadata.get("subsection"))
print("\nUm trecho pronto para o índice:")
print(example.metadata)
print(example.page_content[:200], "...\n")

result = subprocess.run(
    [sys.executable, "-m", "src.rag.ingestion", "--help"], capture_output=True, text=True, check=True
)
print(result.stdout)

shutil.rmtree(WORK_DIR)  # limpeza da pasta temporária da aula
print("Pasta temporária removida:", not WORK_DIR.exists())

chunk_size=1000, chunk_overlap=150
105 documentos -> 321 trechos
pdf         106
markdown     98
html         77
faq          40

Um trecho pronto para o índice:
{'status': 'active', 'audience': 'customers', 'title': 'Política de Cancelamento de Pedidos', 'category': 'policy', 'updated_at': '2026-07-01', 'version': '2', 'source': 'policies/cancellation.md', 'type': 'markdown', 'section': 'Perguntas frequentes', 'subsection': 'Cancelei por engano. Posso desfazer?', 'chunk': 8, 'id': '8fde24d984af07ca'}
[Política de Cancelamento de Pedidos > Perguntas frequentes > Cancelei por engano. Posso desfazer?]
## Perguntas frequentes  
### Cancelei por engano. Posso desfazer?  
Não é possível reativar um pedi ...



usage: ingestion.py [-h] [--incremental] [folder]

Ingestão dos documentos da TechNova no índice FAISS.

positional arguments:
  folder         pasta de documentos (padrão: DOCUMENTS_DIR)

options:
  -h, --help     show this help message and exit
  --incremental  adiciona ao índice existente

Pasta temporária removida: True


## Resumo

- Dividimos os documentos porque um vetor por texto funciona melhor com textos focados: a similaridade com a
  pergunta sobe do manual inteiro para a página e da página para o parágrafo.
- `RecursiveCharacterTextSplitter` corta pelo melhor separador disponível (parágrafo, linha, frase, palavra)
  até caber em `chunk_size`; `chunk_overlap` repete pedaços inteiros do fim do trecho anterior.
- `MarkdownHeaderTextSplitter` divide por título e grava `section` e `subsection`; o splitter recursivo vem
  depois, nas seções grandes.
- O **cabeçalho de contexto** (título > seção > página) no começo de cada trecho tirou do top 4 os trechos de
  manuais errados, por poucos tokens a mais.
- Escolha o `chunk_size` medindo trechos, custo de indexar, acerto da fonte e contexto por pergunta. Na nossa
  base, 1000 é o equilíbrio.
- **Ids estáveis** (hash de fonte + texto) tornam a ingestão idempotente: `add_chunks` ignora o que já existe,
  `remove_source` e `reindex_file` atualizam por fonte. `index_incremental` só adiciona; arquivo editado pede
  `reindex_file`.
- `index_folder()` recria tudo; `python -m src.rag.ingestion` faz o mesmo pela linha de comando.

## Próxima aula

Na aula 2.1 (`02_01_retrievers_and_filters`) começa o módulo 02: com o índice pronto, vamos transformar a busca em
um retriever do LangChain e usar os metadados da ingestão como filtros de atualidade e de acesso.